# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/deepikachebrolu/flyrank-ml/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

### Answer

This lane is a **ranking / scoring** task, not clustering and not a claim about Google's ranking algorithm.

**Decision it supports:** which *visible* pages a human reviewer should inspect first for metadata, snippet/intent, on-page engagement, or monitoring — given limited review time.

**Who acts:** a content or SEO reviewer. The score does not auto-edit a page.

**Wrong-call cost:** a false positive wastes review time; a false negative leaves a high-visibility under-capture page unreviewed. Low-impression pages make both errors more likely because CTR is noisy.

**One-paragraph frame:** For a reviewer deciding which visible pages to inspect first, we will build a ranked review queue from the starter content table (and later warehouse daily facts), scoring a **position- and volume-adjusted CTR/engagement gap**. The primary metric is **Precision@K** on a later 30-day window. On this starter CSV that check is **not fully leakage-safe**: `impressions_90d`, `ctr`, `avg_position`, and `position_tier` overlap the evaluation window. A wrong call costs wasted hours or a missed opportunity. A single if-statement is not enough because CTR depends on position, impression volume, intent, and content type together. We will claim only **observed, directional, decision-support** results — not causality and not a Google ranking factor.

A later optional step can turn a **future observed** under-capture label into classification. That is a validation aid. The product output of this lane is still a ranked list.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

CANDIDATE_PATHS = [
    Path("../../data/raw/content_refresh_anonymized.csv"),
    Path("../data/raw/content_refresh_anonymized.csv"),
    Path("data/raw/content_refresh_anonymized.csv"),
    Path.cwd() / "data" / "raw" / "content_refresh_anonymized.csv",
    Path.cwd().parent / "data" / "raw" / "content_refresh_anonymized.csv",
    Path.cwd().parent.parent / "data" / "raw" / "content_refresh_anonymized.csv",
]
resolved = []
for p in CANDIDATE_PATHS:
    try:
        resolved.append(p.resolve())
    except OSError:
        continue
DATA_PATH = next((p for p in resolved if p.is_file()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Starter CSV not found. Keep data/raw/content_refresh_anonymized.csv in this clone "
        "and run from work/notebooks/ (relative path ../../data/raw/content_refresh_anonymized.csv). "
        "Do not download a substitute file. Warehouse/HF access is not required here."
    )

df = pd.read_csv(DATA_PATH)

print("cwd:", Path.cwd().as_posix())
print("Data file:", DATA_PATH.as_posix())
print("Notebook-dir relative path exists:", Path("../../data/raw/content_refresh_anonymized.csv").exists())
print("Rows:", len(df))
print("Unique content_id:", df["content_id"].nunique())
print("Duplicate content_id rows:", int(df.duplicated("content_id").sum()))
print("Grain check (one row = one content item):", df["content_id"].nunique() == len(df))


MIN_PREV_IMPRESSIONS = 100
eligible_prev = df[df["impressions_prev_30d"] >= MIN_PREV_IMPRESSIONS].copy()


MIN_IMPRESSIONS_90D = 500
visible = df[
    (df["impressions_90d"] >= MIN_IMPRESSIONS_90D)
    & (df["avg_position"] > 0)
    & (df["avg_position"] <= 20)
].copy()

print("Pre-decision eligible (impressions_prev_30d >= 100):", len(eligible_prev))
print(
    "Exploratory 90d visible (impressions_90d >= 500, avg_position in (0, 20]):",
    len(visible),
    "— overlaps last-30; not used to score the chronological check.",
)
print("Task type for this lane: ranking / scoring (ranked review queue)")


cwd: c:/Users/Divya/Documents/flyrank-ml/work/notebooks
Data file: C:/Users/Divya/Documents/flyrank-ml/data/raw/content_refresh_anonymized.csv
Notebook-dir relative path exists: True
Rows: 30000
Unique content_id: 30000
Duplicate content_id rows: 0
Grain check (one row = one content item): True
Pre-decision eligible (impressions_prev_30d >= 100): 18010
Exploratory 90d visible (impressions_90d >= 500, avg_position in (0, 20]): 12023 — overlaps last-30; not used to score the chronological check.
Task type for this lane: ranking / scoring (ranked review queue)


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

### Answer

Three different objects, kept separate:

1. **Observed outcome (preferred label, when a later window exists).**  
   Later-window CTR, built only from later clicks and later impressions, among pages that still have enough impressions. On the starter CSV that later window is `clicks_last_30d / impressions_last_30d` (shown as a ×100 percentage, same convention as `ctr`). On warehouse daily facts it would be clicks/impressions after a decision date. This is measured, not a FlyRank product flag.

2. **Constructed ranking score (what the reviewer sees).**  
   Position-tier CTR residual = observed CTR minus expected CTR among peers in the same `position_tier`, after a minimum-impression filter. More negative residual → higher review priority. This is a **defined score**, not an observed future event. It is the operational ranking output.

3. **Not a target.**  
   `trend_direction` / `trend_pct` / `is_declining_label` (refresh-lane decline label). Product scores such as `health_score` are not in this dataset. Raw `ctr` is also not comparable across rows without position and sample size.

**Scoring fields (pre-decision on this schema):**  
`impressions_prev_30d`, `clicks_prev_30d`, and the prev-30 CTR built from those two. Optional context that does not enter the score in this notebook: `sessions_prev_30d`, `content_type`, `main_intent`, `content_age_days`, `days_since_last_update` / `freshness_tier`, `word_count` (missingness flag; do not blind-fill 0).

**Later-window outcome (not a scoring feature):**  
`clicks_last_30d` / `impressions_last_30d`. Pages without enough last-30 impressions cannot receive this label.

**Overlap that this starter CSV cannot remove:**

- `impressions_90d`, `clicks_90d`, and `ctr` include the last 30 days. They are not eligibility filters and not scoring features for the chronological check.
- `avg_position` is a 90-day mean; `position_tier` is cut from that mean. There is no prev-30 position column. Peer grouping by `position_tier` therefore uses information from the evaluation window. Warehouse daily `gsc_avg_position` before a decision date would fix this.
- Requiring `impressions_last_30d >= 100` to *compute* the label is outcome-window selection: pages that lost volume are dropped from Precision@K, not ranked as unknowns.

This notebook therefore **does not** call the starter evaluation fully leakage-safe.

**Chronological split:** score from prev-30; evaluate last-30 CTR on labeled rows only. Warehouse later: feature window strictly before the decision date; label window after. Do not develop labels on `fact_content_daily_performance_sample` (final month).

In [18]:
MIN_WINDOW_IMPRESSIONS = 100

chrono = eligible_prev[
    (eligible_prev["impressions_prev_30d"] >= MIN_WINDOW_IMPRESSIONS)
    & (eligible_prev["impressions_last_30d"] >= MIN_WINDOW_IMPRESSIONS)
].copy()

chrono["ctr_prev_30d"] = (
    chrono["clicks_prev_30d"]
    / chrono["impressions_prev_30d"]
    * 100
)

chrono["ctr_last_30d"] = (
    chrono["clicks_last_30d"]
    / chrono["impressions_last_30d"]
    * 100
)

expected_prev = chrono["ctr_prev_30d"].median()
chrono["expected_ctr_prev"] = expected_prev
chrono["residual_prev"] = (
    chrono["ctr_prev_30d"] - chrono["expected_ctr_prev"]
)

expected_last = chrono["ctr_last_30d"].median()

chrono["under_capture_last"] = (
    chrono["ctr_last_30d"] < expected_last
).astype(int)

print("Pre-decision eligible pages:", len(eligible_prev))
print("Pages eligible for evaluation:", len(chrono))
print(
    "Later-window under-capture rate:",
    round(chrono["under_capture_last"].mean(), 4)
)
print(
    "Previous-window baseline median CTR:",
    round(expected_prev, 4)
)
print(
    "Evaluation median CTR:",
    round(expected_last, 4)
)


Pre-decision eligible pages: 18010
Pages eligible for evaluation: 15615
Later-window under-capture rate: 0.5
Previous-window baseline median CTR: 0.1434
Evaluation median CTR: 0.1983


## 3. Success metric

*One metric you can defend. What number means 'good'?*

### Answer

**Primary metric: Precision@K** on the chronological split.

- **K** is assumed review capacity. This notebook reports **K = 20** (a shortlist a person can inspect) and **K = 50** (a larger weekly queue). K is not tuned after seeing results.
- A page is **relevant** if its last-30 CTR is below the median last-30 CTR across the eligible evaluation cohort. This is a constructed, relative evaluation label, not a verified business outcome. Pages with fewer than 100 last-30 impressions are excluded from this evaluation, so results apply only to the eligible subset.
- The ranking used here is a **baseline score** (more negative prev-window residual ranks higher). This cell only checks that Precision@K is computable today. It is not a trained-model result.
- **"Good"** means Precision@K is **above the later-window base rate** (the fraction of eligible pages that are relevant). Matching the base rate means the ranking is no better than a random shortlist of size K. Because relevance is "below the eligible-cohort last-window median," that base rate is about 0.5 by construction — so Precision@K near 0.5 is not a win.

**Secondary checks (not substitutes for Precision@K):** count of eligible pages, volume floor, and a hand review of the top 20 reason codes in a later assignment. Accuracy or ROC-AUC on an unbalanced yes/no flag would hide whether the top of the queue is useful.

**What "good" does not mean:** a high Precision@K does not mean a title rewrite caused more clicks.

In [ ]:
def precision_at_k(y_true, scores_higher_is_better, k):
    """Precision@K for a ranked list. scores_higher_is_better: larger = review sooner."""
    k = min(int(k), len(y_true))
    order = np.argsort(-np.asarray(scores_higher_is_better), kind="mergesort")
    top = np.asarray(y_true)[order][:k]
    return float(top.mean()), k

chrono["review_score"] = -chrono["residual_prev"]
base_rate = float(chrono["under_capture_last"].mean())

print("Eligible evaluation pages:", len(chrono))
print("Relevant-page base rate:", round(base_rate, 4))
print()

for k in (20, 50):
    if len(chrono) == 0:
        print(f"Precision@{k}: unavailable — no eligible pages.")
        continue

    p_at_k, k_used = precision_at_k(
        chrono["under_capture_last"],
        chrono["review_score"],
        k
    )

    lift = p_at_k / base_rate if base_rate > 0 else float("nan")

    print(
        f"Precision@{k_used}: {p_at_k:.4f}",
        f"| Base rate: {base_rate:.4f}",
        f"| Lift: {lift:.3f}" if base_rate > 0 else "| Lift: unavailable"
    )

print(
    "\nThis is a transparent baseline-ranking check, not a trained-model result."
)
print(
    "The evaluation excludes pages below the later-window impression threshold."
)
print(
    "The results do not establish that changing a page causes CTR to increase."
)

Eligible evaluation pages: 15615
Relevant-page base rate: 0.5

Precision@20: 0.6500 | Base rate: 0.5000 | Lift: 1.300
Precision@50: 0.6600 | Base rate: 0.5000 | Lift: 1.320

This is a transparent baseline-ranking check, not a trained-model result.
The evaluation excludes pages below the later-window impression threshold.
The results do not establish that changing a page causes CTR to increase.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

### Answer

**One row = one pseudonymized content item (page)** after a visibility filter.

- Eligibility for the scoring exercise is based on at least 100 impressions in the previous 30-day window. The evaluation label uses the subsequent 30-day window and requires at least 100 impressions in that window. Because this excludes low-volume pages based on later-period information, the evaluation is conditional on later-period eligibility and may not represent all pages. The starter CSV does not provide a previous-window position field, so this baseline does not adjust for search position.

In [20]:
unit = chrono[
    [
        "content_id",
        "impressions_prev_30d",
        "clicks_prev_30d",
        "ctr_prev_30d",
        "expected_ctr_prev",
        "residual_prev",
        "impressions_last_30d",
        "clicks_last_30d",
        "ctr_last_30d",
        "under_capture_last",
        "review_score",
    ]
].sort_values("review_score", ascending=False)

print("Unit of analysis: one row = one content item.")
print("Evaluation rows:", len(unit))
print("Unique content IDs:", unit["content_id"].nunique())
print("Duplicate content IDs:", int(unit.duplicated("content_id").sum()))
print()
print("Example ranked rows:")

display_cols = [
    "impressions_prev_30d",
    "ctr_prev_30d",
    "expected_ctr_prev",
    "residual_prev",
    "ctr_last_30d",
    "under_capture_last",
    "review_score",
]

print(unit[display_cols].head(8).round(4).to_string(index=False))


Unit of analysis: one row = one content item.
Evaluation rows: 15615
Unique content IDs: 15615
Duplicate content IDs: 0

Example ranked rows:
 impressions_prev_30d  ctr_prev_30d  expected_ctr_prev  residual_prev  ctr_last_30d  under_capture_last  review_score
                  632           0.0             0.1434        -0.1434        0.1572                   1        0.1434
                  147           0.0             0.1434        -0.1434        0.0000                   1        0.1434
                  356           0.0             0.1434        -0.1434        0.0000                   1        0.1434
                  741           0.0             0.1434        -0.1434        0.0000                   1        0.1434
                  628           0.0             0.1434        -0.1434        0.0000                   1        0.1434
                  739           0.0             0.1434        -0.1434        0.0000                   1        0.1434
                  250           

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

### Answer

A fixed rule such as `if ctr < 0.5 then review` fails for three data reasons already visible in this slice:

1. **Position.** Pages in `top_3` are not comparable to pages in `striking` or `page_1`. Expected CTR differs by `position_tier`, so one global cutoff mixes different jobs.
2. **Sample size.** The same CTR on 500 impressions and on 30,000 impressions is not the same evidence. One extra click moves CTR a lot on small n.
3. **Several signals at once.** Intent, content type, age/freshness, and engagement can sit next to the same position-adjusted CTR gap. An if-statement has to hard-code those combinations; they shift across clients and over time.

What *is* enough as a **baseline** (and must be beaten later): the position-tier residual with a volume floor — a transparent score, not a trained model. ML is only justified if a model ranks the later-window observed label better than that residual on a chronological split, by Precision@K.

This still would not prove that editing a title causes more clicks, and it would not explain Google's ranking system.

In [ ]:

by_tier = (
    visible.groupby("position_tier", observed=True)
    .agg(
        n=("content_id", "size"),
        median_impressions=("impressions_90d", "median"),
        median_ctr=("ctr", "median"),
        p25_ctr=("ctr", lambda s: s.quantile(0.25)),
        p75_ctr=("ctr", lambda s: s.quantile(0.75)),
    )
    .sort_values("median_ctr", ascending=False)
)
print("Visible slice: median CTR still differs by position_tier.")
print("(ctr is a ×100 percentage: 0.76 means 0.76%, not 76%.)")
print(by_tier.round(3).to_string())

print("\nWithin-tier spread (IQR) is large — a single CTR cutoff cannot separate these pages.")

CUTOFF = 0.5
flagged = visible[visible["ctr"] < CUTOFF]
print(
    f"\nGlobal rule ctr < {CUTOFF} flags {len(flagged)} / {len(visible)} visible pages "
    f"({round(len(flagged) / len(visible) * 100, 1)}%)."
)
print("Flag share by position_tier:")
flag_share = (
    visible.assign(flagged=visible["ctr"] < CUTOFF)
    .groupby("position_tier", observed=True)["flagged"]
    .mean()
    .sort_values(ascending=False)
)
print(flag_share.round(3).to_string())
print(
    "\nA later model must beat the position-tier residual baseline on Precision@K; "
    "it is not justified by this exploratory spread alone."
)


Visible slice: median CTR still differs by position_tier.
(ctr is a ×100 percentage: 0.76 means 0.76%, not 76%.)
                  n  median_impressions  median_ctr  p25_ctr  p75_ctr
position_tier                                                        
page_1         7064              4504.0       0.240    0.120    0.440
top_3           458              4263.0       0.200    0.080    0.490
striking       4485              2174.0       0.170    0.080    0.340
page_3_5         16              3482.0       0.155    0.057    0.245

Within-tier spread (IQR) is large — a single CTR cutoff cannot separate these pages.

Global rule ctr < 0.5 flags 9759 / 12023 visible pages (81.2%).
Flag share by position_tier:
position_tier
page_3_5    0.938
striking    0.850
page_1      0.791
top_3       0.751

A later model must beat the position-tier residual baseline on Precision@K; it is not justified by this exploratory spread alone.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

Lane-specific checks:

- [x] Task type is ranking / scoring for a review queue
- [x] Observed later-window CTR is separated from the constructed residual score
- [x] Precision@K is named before any model training; K = 20 and 50
- [x] Unit of analysis is one content item; grain verified on the starter CSV
- [x] Chronological prev-30 vs last-30 split; 90d `ctr` treated as leaky for a last-30 label
- [x] No causal or "predicting Google" claims